# Module 5: Post-Deployment Data Drift Detection & Monitoring
### Course: Applied Data Science & Machine Learning in Production
**Case Study:** Customer Churn & Lifetime Value (LTV) Prediction for a Subscription Business

---
### The MLOps Reality:
In production, machine learning models do not deteriorate because their code breaks—they deteriorate because **the real world changes**. 
- A marketing campaign brings in users with fundamentally different usage patterns (**Covariate Shift / Data Drift**).
- A competitor slashes prices, causing loyal customers to cancel without prior usage drops (**Concept Drift**).

In this module, students build an automated **Statistical Drift Detection Engine** to monitor live telemetry and trigger retraining before model decay causes revenue loss.

### Learning Objectives:
1. **Taxonomy of Distribution Shifts:** Understand Covariate Shift $P(X)$, Prior Probability Shift $P(Y)$, and Concept Drift $P(Y|X)$.
2. **Statistical Drift Testing:** Apply the two-sample **Kolmogorov-Smirnov (KS) test** and **Population Stability Index (PSI)**.
3. **Drift Simulation:** Inject production distribution shifts to stress-test our monitoring engine.
4. **Automated MLOps Governance:** Formulate threshold alerts and retraining triggers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

from pathlib import Path
import sys
sys.path.append("..")
from src.data_loader import load_raw_data, temporal_train_test_split

sns.set_theme(style="whitegrid")
DATA_PATH = Path("../data/raw/subscriptions_raw.csv")
df = load_raw_data(DATA_PATH)
baseline_train, holdout_prod = temporal_train_test_split(df, cutoff_quantile=0.75)

print(f"Baseline (Training) distribution: {len(baseline_train)} records")
print(f"Holdout (Production) distribution: {len(holdout_prod)} records")

## 1. The Kolmogorov-Smirnov (KS) Two-Sample Test
The two-sample KS test compares the empirical cumulative distribution functions (eCDFs) of a baseline feature vs a production monitoring window.
- **Null Hypothesis ($H_0$):** Both samples are drawn from the identical continuous distribution.
- **Decision Rule:** If $p\text{-value} < 0.05$, reject $H_0$ and raise a Data Drift alert.

In [ ]:
# SOLUTION: KS Test Implementation
def evaluate_feature_ks_drift(baseline_series, current_series, alpha=0.05):
    stat, p_value = stats.ks_2samp(baseline_series, current_series)
    is_drifted = p_value < alpha
    return {
        "ks_statistic": round(float(stat), 4),
        "p_value": float(p_value),
        "is_drift_detected": is_drifted
    }

# Check baseline vs normal production holdout
for feature in ["monthly_charges", "usage_drop_pct_last_30d", "tenure_months"]:
    res = evaluate_feature_ks_drift(baseline_train[feature], holdout_prod[feature])
    status_icon = "🚨 DRIFT" if res["is_drift_detected"] else "✓ STABLE"
    print(f"Feature: {feature:25s} | KS Stat: {res['ks_statistic']:.4f} | p-value: {res['p_value']:.4f} | {status_icon}")

## 2. Population Stability Index (PSI)
PSI is the financial industry standard for quantifying distribution shift.
$$\text{PSI} = \sum_{i=1}^{k} \Big(P_i - Q_i\Big) \times \ln\left(\frac{P_i}{Q_i}\right)$$
Where $P_i$ is the actual production frequency and $Q_i$ is the expected baseline frequency in bin $i$.
- **$\text{PSI} < 0.10$:** Negligible shift (Model is healthy).
- **$0.10 \le \text{PSI} < 0.25$:** Moderate drift (Investigate).
- **$\text{PSI} \ge 0.25$:** Significant drift (Immediate retraining required).

In [ ]:
# SOLUTION: Population Stability Index (PSI) Calculation
def calculate_psi(expected, actual, num_bins=10):
    # Generate quantile bins from baseline (expected)
    percentiles = np.linspace(0, 100, num_bins + 1)
    bin_edges = np.percentile(expected, percentiles)
    bin_edges[0] -= 1e-5
    bin_edges[-1] += 1e-5
    
    # Calculate frequencies
    expected_counts, _ = np.histogram(expected, bins=bin_edges)
    actual_counts, _ = np.histogram(actual, bins=bin_edges)
    
    # Proportions with small epsilon to prevent div by zero
    eps = 1e-4
    P = (actual_counts / len(actual)) + eps
    Q = (expected_counts / len(expected)) + eps
    
    psi_val = np.sum((P - Q) * np.log(P / Q))
    return float(psi_val)

psi_usage = calculate_psi(baseline_train["usage_drop_pct_last_30d"], holdout_prod["usage_drop_pct_last_30d"])
print(f"Baseline vs Holdout PSI (usage_drop_pct_last_30d): {psi_usage:.4f} (Status: {'STABLE' if psi_usage < 0.1 else 'DRIFT'})")

## 3. Simulating a Real-World Production Covariate Shift
Imagine CloudStream launched a viral marketing campaign targeting high-usage users whose engagement suddenly drops after a promotion ends.

In [ ]:
drifted_prod = holdout_prod.copy()
# Artificially simulate a heavy distribution shift in usage drop
rng = np.random.default_rng(99)
drifted_prod["usage_drop_pct_last_30d"] += rng.normal(0.40, 0.15, size=len(drifted_prod))

# Compare distributions visually
plt.figure(figsize=(10, 5))
sns.kdeplot(baseline_train["usage_drop_pct_last_30d"], label="Baseline (Training) Distribution", color="blue", fill=True, alpha=0.3)
sns.kdeplot(drifted_prod["usage_drop_pct_last_30d"], label="Simulated Drifted Telemetry", color="red", fill=True, alpha=0.3)
plt.title("Covariate Shift Detection: 30-Day Usage Drop Telemetry", fontsize=14)
plt.xlabel("Usage Drop Percentage")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
res_drift = evaluate_feature_ks_drift(baseline_train["usage_drop_pct_last_30d"], drifted_prod["usage_drop_pct_last_30d"])
psi_drift = calculate_psi(baseline_train["usage_drop_pct_last_30d"], drifted_prod["usage_drop_pct_last_30d"])

print("=== MLOps Monitoring Alert Summary ===")
print(f"KS Statistic:   {res_drift['ks_statistic']:.4f} (p-value: {res_drift['p_value']:.4e})")
print(f"PSI Score:      {psi_drift:.4f}")
if psi_drift >= 0.25:
    print("🚨 CRITICAL ALERT: Significant Data Drift Detected! Triggering Automated Retraining Pipeline.")
else:
    print("✓ Model inputs stable.")

## 4. Automated Governance & CI/CD Pipeline Integration
In production, this monitoring logic is exposed via our FastAPI endpoint (`POST /telemetry/drift`). 
Scheduled GitHub Actions workflows or cron jobs query this endpoint to proactively trigger retraining before churn predictions diverge from reality.